# Theme 2: RAG Application, Starter Kit

This starter kit contains a complete, working **Retrieval-Augmented Generation (RAG)** application using **Gemma 4**, Google's open-weight model. Your team's task is to adapt and extend it for the problem statement you selected (2A or 2B).

**How RAG works:**

```
 Document ──► 1. split into chunks ──► 2. convert each chunk into an embedding (a list of numbers)
                                                           │
 Question ──► 3. retrieve the chunks whose embeddings are closest to the question's embedding
                                                           │
              4. give ONLY those chunks to Gemma ──► answer with page citations
```

**Models used (both open-weight):**
- `all-MiniLM-L6-v2` (sentence-transformers) creates the embeddings. It runs inside this notebook.
- `gemma-4-26b-a4b-it` (Google) writes the answers. It is accessed through the free Google AI Studio API.

**No GPU is required.** Use the default Colab runtime (CPU).

### Obtaining a free Google AI Studio API key (one per team)
1. Go to **https://aistudio.google.com/apikey** and sign in with a Google account.
2. Select **Create API key** and copy the key.
3. In Colab, select the **key icon** in the left sidebar (**Secrets**).
4. Select **Add new secret**. Set **Name** to `GOOGLE_API_KEY` and **Value** to your key. Enable **Notebook access**.

> **Important:** do not paste the key directly into a code cell. Anyone the notebook is shared with would be able to see it.
>
> **No key yet:** the notebook still runs. The retrieval stages work normally, and the application shows the passages it found in place of a written answer.

### How to use this notebook
- Run the cells **in order, from top to bottom**, the first time (select a cell and press **Shift + Enter**).
- Comments marked **STUDENT TODO** show the intended places to make changes.
- After changing a cell, re-run that cell and any cells below it that depend on it.
- If the notebook stops working as expected, select **Runtime → Restart session** and run all cells again from the top.

## Step 0: Installation (run once)

In [ ]:
# Installs the required libraries. This takes about one minute.
!pip install -q pypdf sentence-transformers "gradio>=6,<7" google-genai

## Step 1: Settings

These values control how documents are divided, searched and answered. They are the main parameters to experiment with.

In [ ]:

import os

print("API key available:", bool(os.environ.get("GOOGLE_API_KEY")))
print("Model configured:", bool(globals().get("GEMMA_MODEL")))
print("Demo variable exists:", "DEMO_MODE" in globals())

API key available: True
Model configured: True
Demo variable exists: True


In [ ]:
# STUDENT TODO: the Gemma 4 model that writes the answers.
#   "gemma-4-26b-a4b-it"  Mixture of Experts, 26B total / about 4B active parameters. Fast. (default)
#   "gemma-4-31b-it"      Dense, 31B parameters. More capable, slower.
GEMMA_MODEL = "gemma-4-26b-a4b-it"

# STUDENT TODO: the number of words in each chunk.
#   Small chunks (50)   more precise matches, but may lose context
#   Large chunks (300)  more context, but less precise and longer prompts
CHUNK_SIZE = 120

# STUDENT TODO: the number of words shared by neighbouring chunks, so sentences are not split.
CHUNK_OVERLAP = 30

# STUDENT TODO: the number of chunks given to Gemma for each question.
TOP_K = 3

# STUDENT TODO: the rules Gemma follows when answering. The rule to use only the context prevents invented answers.
ANSWER_RULES = """You answer questions about a document using ONLY the context passages provided.
- Each passage begins with its page number, for example [Page 2].
- After each fact you use, cite the page in brackets, for example (p. 2).
- If the answer is not in the context, reply exactly: "The answer could not be found in the document."
- Keep answers concise: 2 to 4 sentences."""

TEMPERATURE = 0.2       # a low value keeps answers consistent and close to the document

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # STUDENT TODO (optional): try "BAAI/bge-small-en-v1.5"
print("Settings configured.")

Settings configured.


## Step 2: Connect to Gemma and load the embedding model

In [ ]:
import os
from google import genai
from google.genai import types, errors


def _get_secret(name):
    """Reads a secret from the Colab Secrets panel (or from an environment variable)."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


GOOGLE_API_KEY = _get_secret("GOOGLE_API_KEY")
DEMO_MODE = not GOOGLE_API_KEY
client = None if DEMO_MODE else genai.Client(api_key=GOOGLE_API_KEY)


def _friendly_error(e):
    code = getattr(e, "code", None)
    if code in (400, 401, 403) and "key" in str(e).lower():
        return "Error: the GOOGLE_API_KEY was rejected. Check that the complete key is saved in Colab Secrets, then re-run this cell."
    if code == 429:
        return "Error: the free-tier usage limit has been reached. Wait about one minute and try again."
    if code == 404:
        return f"Error: the model '{GEMMA_MODEL}' was not found. Check the GEMMA_MODEL setting."
    return f"Error: the request to Gemma failed ({e})."


def generate(user_text, system_text=None, temperature=TEMPERATURE, demo_reply="(Demonstration mode response)"):
    """Sends one message to Gemma and returns its reply as text.
    In demonstration mode, returns `demo_reply`. If the request fails, returns a message beginning with "Error:"."""
    if DEMO_MODE:
        return demo_reply
    try:
        response = client.models.generate_content(
            model = GEMMA_MODEL,
            contents = user_text,
            config = types.GenerateContentConfig(
                system_instruction = system_text,
                temperature = temperature,
                thinking_config = types.ThinkingConfig(thinking_level="minimal"),  # faster replies
            ),
        )
        return (response.text or "").strip()
    except errors.APIError as e:
        return _friendly_error(e)
    except Exception as e:
        return f"Error: {e}"


if DEMO_MODE:
    print("Demonstration mode: no GOOGLE_API_KEY was found. Retrieval will work; answers will show the passages found.")
else:
    print(f"Connected to Google AI Studio. Model: {GEMMA_MODEL}")
    print(generate("In one sentence, what is an open-weight AI model?"))

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL_NAME)   # runs on the CPU
print(f"Loaded embedding model: {EMBED_MODEL_NAME}")

Connected to Google AI Studio. Model: gemma-4-26b-a4b-it
An open-weight AI model is a machine learning model where the trained parameters (weights) are made publicly available, allowing developers to run, study, and fine-tune the model on their own hardware.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2


In [ ]:
from google.colab import userdata
import os

api_key = userdata.get("GOOGLE_API_KEY")

if api_key:
    os.environ["GOOGLE_API_KEY"] = api_key
    print("API key detected successfully!")
else:
    print("API key not found. Check Colab Secrets.")

API key detected successfully!


## Step 3: Sample document
A short, fictional course syllabus is provided so the application works immediately. Each item in the list is one page. Your own PDF can be uploaded in the application in Step 5.

In [ ]:
# STUDENT TODO: replace with text relevant to your problem statement, or upload a PDF in the application instead.
SAMPLE_PAGES = [
    """CS 101: Introduction to Programming (Sample Syllabus). Instructor: Dr. A. Rao. Email: arao@example.edu.
    Lectures are held Monday and Wednesday, 10:00 to 11:30 AM, in Room B-204. Labs are on Friday from 2:00 to 4:00 PM
    in the Computer Lab (Room C-110). Office hours: Tuesday 3:00 to 5:00 PM, or by appointment. This course introduces
    programming using Python. No prior coding experience is required.""",

    """Grading. Weekly lab exercises: 20%. Two mini-projects: 30% (15% each). Midterm exam: 20%. Final exam: 25%.
    Class participation: 5%. A score of 90% or above earns an A, 80 to 89% a B, 70 to 79% a C, and 60 to 69% a D.
    Late submissions lose 10% per day, up to a maximum of 3 days, after which they receive zero. Each student has
    two grace days per semester that may be used to submit any lab late without penalty.""",

    """Schedule. Weeks 1-2: variables, data types, and input/output. Weeks 3-4: conditions and loops. Week 5: functions.
    Week 6: lists and dictionaries. Week 7: Midterm exam (in class, closed book). Weeks 8-9: files and error handling.
    Weeks 10-11: an introduction to APIs and working with data. Weeks 12-13: mini-project 2, a team project of up to
    3 students. Week 14: review. The final exam is a 2-hour written exam during the examination period.""",

    """Policies. Attendance: students must attend at least 75% of lectures to be eligible for the final exam.
    Academic integrity: students may discuss ideas with classmates, but all submitted code must be their own. AI
    tools may be used for learning and debugging, but their use must be stated in a comment at the top of the file.
    Laptops: bring a laptop to every lab. Students without a laptop may reserve a machine in the Computer Lab.
    Textbook: "Automate the Boring Stuff with Python" (available free online).""",
]
print(f"Sample document contains {len(SAMPLE_PAGES)} pages.")

Sample document contains 4 pages.


## Step 4: The RAG pipeline
Each function is one stage of the pipeline. This is the core of the project and the main code to extend.

In [ ]:
import numpy as np
from pypdf import PdfReader

knowledge_base = {"name": None, "chunks": [], "vectors": None}  # populated by build_knowledge_base()


def read_pdf(path):
    """Returns a list of page texts from a PDF file."""
    return [page.extract_text() or "" for page in PdfReader(path).pages]


def split_into_chunks(pages):
    """STAGE 1: split each page into overlapping chunks of about CHUNK_SIZE words."""
    chunks = []
    step = max(1, CHUNK_SIZE - CHUNK_OVERLAP)
    for page_number, text in enumerate(pages, start=1):
        words = text.split()
        for start in range(0, len(words), step):
            chunk_words = words[start:start + CHUNK_SIZE]
            if len(chunk_words) > 5:  # skip very short fragments
                chunks.append({"page": page_number, "text": " ".join(chunk_words)})
            if start + CHUNK_SIZE >= len(words):
                break
    return chunks


def build_knowledge_base(pages, name):
    """STAGE 2: convert each chunk into an embedding."""
    chunks = split_into_chunks(pages)
    if not chunks:
        raise ValueError("No text was found. The PDF may contain scanned images only.")
    vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
    knowledge_base.update(name=name, chunks=chunks, vectors=vectors)
    return f"Loaded **{name}**: {len(pages)} pages, {len(chunks)} chunks."


def retrieve(question, top_k=TOP_K):
    """STAGE 3: find the chunks whose meaning is closest to the question."""
    question_vector = embedder.encode([question], normalize_embeddings=True)[0]
    similarities = knowledge_base["vectors"] @ question_vector   # 1.0 = identical meaning
    best = np.argsort(similarities)[::-1][:top_k]
    return [{**knowledge_base["chunks"][i], "score": float(similarities[i])} for i in best]


def answer(question):
    """STAGE 4: give the retrieved chunks to Gemma and ask it to answer using only them."""
    found = retrieve(question)
    context = "\n\n".join(f"[Page {c['page']}] {c['text']}" for c in found)
    prompt = f"Context:\n{context}\n\nQuestion: {question}"
    demo = (f"**Demonstration mode:** no API key is set, so Gemma did not write an answer. Retrieval completed; "
            f"the most relevant passage is on **page {found[0]['page']}**:\n\n> {found[0]['text'][:400]}...")
    reply = generate(prompt, system_text=ANSWER_RULES, demo_reply=demo)
    return reply, found


# Quick test with the sample document.
print(build_knowledge_base(SAMPLE_PAGES, "Sample CS 101 syllabus"))
test_question = "What happens if I submit a lab late?"
reply, found = answer(test_question)
print(f"\nQuestion: {test_question}\nAnswer: {reply}\n\nRetrieved passages:")
for c in found:
    print(f"   [page {c['page']}, similarity {c['score']:.2f}] {c['text'][:90]}...")

Loaded **Sample CS 101 syllabus**: 1 pages, 1 chunks.

Question: What happens if I submit a lab late?
Answer: The answer could not be found in the document.

Retrieved passages:
   [page 1, similarity 0.07] OpenBridge is a guide for people who want to contribute to open-source projects. Before co...


In [ ]:
import os

doc_paths = [
    "/content/README.md",
    "/content/CONTRIBUTING.md"
]

for path in doc_paths:
    if os.path.exists(path):
        print("✅ Found:", os.path.basename(path))
    else:
        print("❌ Missing:", path)


✅ Found: README.md
✅ Found: CONTRIBUTING.md


In [ ]:
import os

def read_markdown(path):
    with open(path, "r", encoding="utf-8") as file:
        return file.read()

documents = {}

for path in [
    "/content/README.md",
    "/content/CONTRIBUTING.md"
]:
    if os.path.exists(path):
        documents[os.path.basename(path)] = read_markdown(path)

for name, content in documents.items():
    print(f"\n📄 {name}")
    print("Characters:", len(content))
    print(content[:300])


📄 README.md
Characters: 6836
[![freeCodeCamp Social Banner](https://cdn.freecodecamp.org/platform/universal/fcc_banner_new.png)](https://www.freecodecamp.org/)

[![first-timers-only Friendly](https://img.shields.io/badge/first--timers--only-friendly-blue.svg)](https://www.firsttimersonly.com/)
[![Discord](https://img.shields.io

📄 CONTRIBUTING.md
Characters: 9826
# Contributor's Guide

'HOW TO CONTRIBUTE TO OPEN SOURCE' accepts PR's (pull requests) from **newbies**
only. This is to help **newbies** get familiar with the contribution processes.

Issues can be submitted by anyone - either seasoned developers or newbies.

**Contents**

- [Choosing an Issue](#ch


In [ ]:
SAMPLE_PAGES = [
    """
    OpenBridge is a guide for people who want to contribute
    to open-source projects.

    Before contributing, read the project's README file
    and CONTRIBUTING guide.

    Follow the setup instructions to install the project.

    Read the Code of Conduct before interacting with
    the project's community.

    If you find a bug, check the project's documentation
    for instructions on reporting issues.
    """
]

print("Sample document created!")
print("Number of pages:", len(SAMPLE_PAGES))

Sample document created!
Number of pages: 1


In [ ]:
print(build_knowledge_base(SAMPLE_PAGES, "Sample CS 101 syllabus"))
test_question = "What happens if I submit a lab late?"
reply, found = answer(test_question)

Loaded **Sample CS 101 syllabus**: 1 pages, 1 chunks.


In [ ]:
print(build_knowledge_base(SAMPLE_PAGES, "OpenBridge Sample Guide"))

test_question = "What should a beginner read before contributing?"
reply, found = answer(test_question)

print(f"\nQuestion: {test_question}")
print(f"\nAnswer: {reply}")
print("\nRetrieved passages:")

for c in found:
    print(
        f"Page {c['page']} | "
        f"Similarity: {c['score']:.2f} | "
        f"{c['text'][:200]}"
    )

Loaded **OpenBridge Sample Guide**: 1 pages, 1 chunks.

Question: What should a beginner read before contributing?

Answer: Error: the request to Gemma failed (500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}).

Retrieved passages:
Page 1 | Similarity: 0.33 | OpenBridge is a guide for people who want to contribute to open-source projects. Before contributing, read the project's README file and CONTRIBUTING guide. Follow the setup instructions to install th


In [ ]:


chunk_size = 500
overlap = 100

chunks = []

for filename, content in documents.items():
    content = content.strip()

    start = 0
    while start < len(content):
        end = start + chunk_size
        chunk_text = content[start:end]

        if chunk_text.strip():
            chunks.append({
                "source": filename,
                "text": chunk_text
            })

        start += chunk_size - overlap

print("Total chunks created:", len(chunks))

for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i + 1}")
    print("Source:", chunk["source"])
    print(chunk["text"][:200])

Total chunks created: 43

Chunk 1
Source: README.md
[![freeCodeCamp Social Banner](https://cdn.freecodecamp.org/platform/universal/fcc_banner_new.png)](https://www.freecodecamp.org/)

[![first-timers-only Friendly](https://img.shields.io/badge/first--t

Chunk 2
Source: README.md

[![LFX Active Contributors](https://insights.linuxfoundation.org/api/badge/active-contributors?project=freecodecamp&repos=https://github.com/freeCodeCamp/freeCodeCamp)](https://insights.linuxfoundati

Chunk 3
Source: README.md
here you can learn to code for free. It is run by a [donor-supported 501(c)(3) charity](https://www.freecodecamp.org/donate) to help millions of busy adults transition into tech. Our community has alr


In [ ]:


!pip -q install sentence-transformers

from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Total chunks:", len(chunks))
print("Embedding shape:", embeddings.shape)
print("Embeddings created successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Total chunks: 43
Embedding shape: (43, 384)
Embeddings created successfully!


In [ ]:
def retrieve_documents(question, top_k=3):
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    scores = embeddings @ question_embedding
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append({
            "source": chunks[idx]["source"],
            "text": chunks[idx]["text"],
            "score": float(scores[idx])
        })

    return results


question = "How can I contribute to this open-source project?"
results = retrieve_documents(question)

for i, result in enumerate(results, 1):
    print(f"\nResult {i} | Source: {result['source']}")
    print(f"Similarity: {result['score']:.3f}")
    print(result["text"][:300])


Result 1 | Source: CONTRIBUTING.md
Similarity: 0.690
ne, feel free to
[create an issue](https://github.com/freeCodeCamp/how-to-contribute-to-open-source/issues/new/choose)
to crowdsource help.

### Adding to the Project File

We have a
[`PROJECT.md`](https://github.com/freeCodeCamp/how-to-contribute-to-open-source/blob/master/PROJECTS.md)
file to help

Result 2 | Source: CONTRIBUTING.md
Similarity: 0.672
n-source/blob/master/README.md)
contains a list of useful resources for beginners who want to contribute to
open source.

You can contribute to this page by adding a Markdown-formatted link.

It should look similar to the one below.

```
- [Title of the page](www.websitename.com/slug-name-here) - Ad

Result 3 | Source: CONTRIBUTING.md
Similarity: 0.646
 How to Contribute"](https://www.youtube.com/watch?v=yzeVMecydCE) (41:52) - Learn why and how to contribute to open source software with Eddie Jaoude.
```

### Adding to Non-English README

The main `README.md` file is written in Englis

In [ ]:
question = "How can I contribute to this open-source project?"

results = retrieve_documents(question, top_k=3)

for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Source:", result["source"])
    print("Similarity:", round(result["score"], 3))
    print("Text:", result["text"][:400])


--- Result 1 ---
Source: CONTRIBUTING.md
Similarity: 0.69
Text: ne, feel free to
[create an issue](https://github.com/freeCodeCamp/how-to-contribute-to-open-source/issues/new/choose)
to crowdsource help.

### Adding to the Project File

We have a
[`PROJECT.md`](https://github.com/freeCodeCamp/how-to-contribute-to-open-source/blob/master/PROJECTS.md)
file to help curate a list of projects and organizations that are friendly to contributions.

If you know of a p

--- Result 2 ---
Source: CONTRIBUTING.md
Similarity: 0.672
Text: n-source/blob/master/README.md)
contains a list of useful resources for beginners who want to contribute to
open source.

You can contribute to this page by adding a Markdown-formatted link.

It should look similar to the one below.

```
- [Title of the page](www.websitename.com/slug-name-here) - Add a description of why I should look at this site.
```

When in doubt, take a look at the current li

--- Result 3 ---
Source: CONTRIBUTING.md
Similarity: 0.646
Text:  

In [ ]:
def generate_grounded_answer(question, results):
    context = "\n\n".join(
        f"Source: {item['source']}\n{item['text']}"
        for item in results
    )

    prompt = f"""
You are OpenBridge, an assistant helping beginners
contribute to open-source projects.

Answer the question using ONLY the context below.
If the context does not contain the answer, say:
"I couldn't find this information in the provided documents."

Cite the source filename in your answer.
Do not invent repository rules or instructions.

CONTEXT:
{context}

QUESTION:
{question}

Answer with source filenames:
"""

    return generate(prompt)


question = "How can I contribute to this open-source project?"
results = retrieve_documents(question, top_k=3)

answer_text = generate_grounded_answer(question, results)
print(answer_text)

You can contribute to this project in the following ways:

*   **Crowdsourcing help:** You can [create an issue](https://github.com/freeCodeCamp/how-to-contribute-to-open-source/issues/new/choose) to crowdsource help. (Source: CONTRIBUTING.md)
*   **Adding to the Project File:** You can add projects or organizations that are friendly to contributions to the [`PROJECT.md`](https://github.com/freeCodeCamp/how-to-contribute-to-open-source/blob/master/PROJECTS.md) file. (Source: CONTRIBUTING.md)
*   **Adding to the README:** You can contribute to the list of useful resources for beginners by adding a Markdown-formatted link. It should follow this format: `- [Title of the page](www.websitename.com/slug-name-here) - Add a description of why I should look at this site.` (Source: CONTRIBUTING.md)
*   **Non-English README:** While the main `README.md` is in English and serves as a template, it is recommended that you provide language-specific resources links instead of translating. (Source: CON

## Step 5: Launch the application
Run this cell to open the application below the cell and create a **public link** (`https://....gradio.live`). Re-run it after changing any cell above.

In [ ]:
import gradio as gr

gr.close_all()  # closes any previous version of the application that is still running

APP_TITLE = "Document Question Answering"  # STUDENT TODO: rename for your problem statement
APP_DESCRIPTION = "Upload a PDF and ask questions. Answers are based only on the document and include page references."


def on_upload(file):
    if file is None:
        return "No file uploaded."
    try:
        return build_knowledge_base(read_pdf(file), os.path.basename(file))
    except Exception as e:
        raise gr.Error(f"The PDF could not be read: {e}")


def on_ask(question):
    if not question or not question.strip():
        raise gr.Error("Enter a question first.")

    try:
        found = retrieve_documents(question, top_k=3)

        if not found:
            return (
                "I couldn't find relevant information in the provided documents.",
                "No relevant passages found."
            )

        reply = generate_grounded_answer(question, found)

        sources = "\n\n".join(
            f"**Source: {item['source']}** "
            f"(similarity: {item['score']:.2f})\n\n"
            f"{item['text']}"
            for item in found
        )

        return reply, sources

    except Exception as e:
        raise gr.Error(f"OpenBridge could not generate an answer: {e}")


with gr.Blocks(title=APP_TITLE) as demo:
    gr.Markdown(f"# {APP_TITLE}\n{APP_DESCRIPTION}")
    with gr.Row():
        pdf_input = gr.File(label="Upload a PDF", file_types=[".pdf"], type="filepath")
        with gr.Column():
            status_md = gr.Markdown(f"Current document: **{knowledge_base['name']}**")
            sample_btn = gr.Button("Use the sample document")
    question_box = gr.Textbox(label="Question", placeholder="For example: How is the course graded?")
    ask_btn = gr.Button("Ask", variant="primary")
    gr.Examples(["How is the course graded?", "When are office hours?", "Can I use AI tools?",
                 "Who won the 2022 World Cup?"], inputs=question_box)  # STUDENT TODO: questions for your documents
    answer_md = gr.Markdown()
    with gr.Accordion("Retrieved passages (the text given to Gemma)", open=False):
        sources_md = gr.Markdown()

    pdf_input.upload(on_upload, inputs=pdf_input, outputs=status_md)
    sample_btn.click(lambda: build_knowledge_base(SAMPLE_PAGES, "Sample CS 101 syllabus"), outputs=status_md)
    ask_btn.click(on_ask, inputs=question_box, outputs=[answer_md, sources_md])
    question_box.submit(on_ask, inputs=question_box, outputs=[answer_md, sources_md])

demo.launch(share=True)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f576dfa0e4c189da6e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
question = "What is the process for reporting an issue?"

results = retrieve_documents(question, top_k=5)

for i, item in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Source:", item["source"])
    print("Similarity:", round(item["score"], 3))
    print(item["text"][:700])


--- Result 1 ---
Source: CONTRIBUTING.md
Similarity: 0.429
e an issue to work on. Here is a basic workflow you want to work from:

1. Search through issues
2. Find issue you want to work on
3. Check if someone else has already worked on and made a pull request on said issue
4. (Optional) Double check pull requests for someone who has worked on the pull request

If you have gotten that far, then you can go ahead and work on the issue. Below are more detailed instructions based on the basic workflow above.

You can find open issue [here](https://github.co

--- Result 2 ---
Source: CONTRIBUTING.md
Similarity: 0.417
s somebody already working on a fix. If no one is working on it, kindly leave a comment stating that you intend to work on it. By doing that,
other people don't accidentally duplicate your effort.

In a situation where somebody decides to fix an issue but doesn't follow up
for a particular period of time, say 2-3 weeks, it's acceptable to still pick
up the issue but make sure 

In [ ]:
import re

chunks = []

for filename, content in documents.items():
    content = content.strip()

    # Split on blank lines, preserving meaningful paragraphs
    paragraphs = re.split(r"\n\s*\n", content)

    current_chunk = ""

    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue

        if len(current_chunk) + len(paragraph) + 2 <= 1000:
            current_chunk += ("\n\n" if current_chunk else "") + paragraph
        else:
            if current_chunk:
                chunks.append({
                    "source": filename,
                    "text": current_chunk
                })
            current_chunk = paragraph

    if current_chunk:
        chunks.append({
            "source": filename,
            "text": current_chunk
        })

print("Improved chunks:", len(chunks))
print("Sample chunk:")
print(chunks[0]["text"][:500])

Improved chunks: 20
Sample chunk:
[![freeCodeCamp Social Banner](https://cdn.freecodecamp.org/platform/universal/fcc_banner_new.png)](https://www.freecodecamp.org/)

[![first-timers-only Friendly](https://img.shields.io/badge/first--timers--only-friendly-blue.svg)](https://www.firsttimersonly.com/)
[![Discord](https://img.shields.io/discord/692816967895220344?logo=discord&label=Discord&color=5865F2)](https://discord.gg/PRyKn3Vbay)
[![LFX Active Contributors](https://insights.linuxfoundation.org/api/badge/active-contributors?proj


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the embedding model if it isn't already loaded
if "embedding_model" not in globals():
    embedding_model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )

# Create fresh embeddings from the improved chunks
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Total improved chunks:", len(chunks))
print("Embedding shape:", embeddings.shape)
print("Embeddings rebuilt successfully!")

Total improved chunks: 20
Embedding shape: (20, 384)
Embeddings rebuilt successfully!


In [ ]:
question = "What is the process for reporting an issue?"

results = retrieve_documents(question, top_k=5)

for i, item in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Source:", item["source"])
    print("Similarity:", round(item["score"], 3))
    print(item["text"][:700])


--- Result 1 ---
Source: CONTRIBUTING.md
Similarity: 0.394
If you decide to fix an issue, it's advisable to check the comment thread to see if there's somebody already working on a fix. If no one is working on it, kindly leave a comment stating that you intend to work on it. By doing that,
other people don't accidentally duplicate your effort.

In a situation where somebody decides to fix an issue but doesn't follow up
for a particular period of time, say 2-3 weeks, it's acceptable to still pick
up the issue but make sure that you leave a comment.

*Note*: Every open-source project has a **CONTRIBUTING.md** file, please make
sure to read this before you open up a pull request; otherwise, it may be
rejected. However, if you do not see any CONTRIBUTIN

--- Result 2 ---
Source: CONTRIBUTING.md
Similarity: 0.298
1. Search through issues
2. Find issue you want to work on
3. Check if someone else has already worked on and made a pull request on said issue
4. (Optional) Double check pull req

In [ ]:
question = "What is the process for reporting an issue?"

results = retrieve_documents(question, top_k=5)

answer_text = generate_grounded_answer(question, results)

print("QUESTION:", question)
print("\nANSWER:\n")
print(answer_text)

print("\nSOURCES USED:")
for item in results:
    print("-", item["source"])

QUESTION: What is the process for reporting an issue?

ANSWER:

I couldn't find this information in the provided documents.

SOURCES USED:
- CONTRIBUTING.md
- CONTRIBUTING.md
- CONTRIBUTING.md
- CONTRIBUTING.md
- README.md


In [ ]:
bug_reporting_guide = """
OpenBridge: Bug Reporting Guide

Before reporting a bug:
1. Search existing issues to check whether the bug
   has already been reported.
2. Check the project's README and CONTRIBUTING guide
   for issue-reporting instructions.
3. If a similar issue exists, add useful details there
   instead of creating a duplicate.

When reporting a new bug:
1. Open the project's GitHub Issues page.
2. Check whether the issue is reproducible.
3. Describe what you expected to happen.
4. Describe what actually happened.
5. Include steps to reproduce the bug.
6. Include relevant error messages and environment details.
7. Follow the project's issue template and contribution rules.

Important:
These are general reporting guidelines for OpenBridge.
Always check the target project's official documentation
for its specific reporting process.
"""

documents["OpenBridge_Bug_Reporting_Guide.md"] = bug_reporting_guide

print("Added bug reporting guide.")
print("Total documents:", len(documents))

Added bug reporting guide.
Total documents: 3


In [ ]:
import re

chunks = []

for filename, content in documents.items():
    content = content.strip()
    paragraphs = re.split(r"\n\s*\n", content)
    current_chunk = ""

    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue

        if len(current_chunk) + len(paragraph) + 2 <= 1000:
            current_chunk += ("\n\n" if current_chunk else "") + paragraph
        else:
            if current_chunk:
                chunks.append({
                    "source": filename,
                    "text": current_chunk
                })
            current_chunk = paragraph

    if current_chunk:
        chunks.append({
            "source": filename,
            "text": current_chunk
        })

print("Total documents:", len(documents))
print("Total chunks:", len(chunks))
print("New guide included:", any(
    c["source"] == "OpenBridge_Bug_Reporting_Guide.md"
    for c in chunks
))

Total documents: 3
Total chunks: 21
New guide included: True


In [ ]:
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings refreshed:", embeddings.shape)


Embeddings refreshed: (21, 384)


In [ ]:
question = "How can I contribute to this open-source project?"

results = retrieve_documents(question, top_k=3)

print("Retrieved passages:", len(results))

for item in results:
    print("\nSource:", item["source"])
    print(item["text"][:200])

print("\nTesting Gemma directly...")
print(generate("Reply with exactly: OpenBridge test successful"))

Retrieved passages: 3

Source: CONTRIBUTING.md
ne, feel free to
[create an issue](https://github.com/freeCodeCamp/how-to-contribute-to-open-source/issues/new/choose)
to crowdsource help.

### Adding to the Project File

We have a
[`PROJECT.md`](ht

Source: CONTRIBUTING.md
n-source/blob/master/README.md)
contains a list of useful resources for beginners who want to contribute to
open source.

You can contribute to this page by adding a Markdown-formatted link.

It shoul

Source: CONTRIBUTING.md
 How to Contribute"](https://www.youtube.com/watch?v=yzeVMecydCE) (41:52) - Learn why and how to contribute to open source software with Eddie Jaoude.
```

### Adding to Non-English README

The main `

Testing Gemma directly...
OpenBridge test successful


In [ ]:
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Total chunks:", len(chunks))
print("Embeddings shape:", embeddings.shape)
print("Embeddings refreshed successfully!")

Total chunks: 21
Embeddings shape: (21, 384)
Embeddings refreshed successfully!


In [ ]:
question = "What is the process for reporting an issue?"

results = retrieve_documents(question, top_k=5)

for i, item in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Source:", item["source"])
    print("Similarity:", round(item["score"], 3))
    print(item["text"][:500])


--- Result 1 ---
Source: OpenBridge_Bug_Reporting_Guide.md
Similarity: 0.398
OpenBridge: Bug Reporting Guide

Before reporting a bug:
1. Search existing issues to check whether the bug
   has already been reported.
2. Check the project's README and CONTRIBUTING guide
   for issue-reporting instructions.
3. If a similar issue exists, add useful details there
   instead of creating a duplicate.

When reporting a new bug:
1. Open the project's GitHub Issues page.
2. Check whether the issue is reproducible.
3. Describe what you expected to happen.
4. Describe what actually h

--- Result 2 ---
Source: CONTRIBUTING.md
Similarity: 0.394
If you decide to fix an issue, it's advisable to check the comment thread to see if there's somebody already working on a fix. If no one is working on it, kindly leave a comment stating that you intend to work on it. By doing that,
other people don't accidentally duplicate your effort.

In a situation where somebody decides to fix an issue but doesn't follow 

In [ ]:
# Check whether the new guide exists in the chunks
guide_chunks = [
    chunk for chunk in chunks
    if chunk["source"] == "OpenBridge_Bug_Reporting_Guide.md"
]

print("Guide chunks found:", len(guide_chunks))

for i, chunk in enumerate(guide_chunks, 1):
    print(f"\nGuide chunk {i}:")
    print(chunk["text"][:300])

# Check the retrieval function's results
question = "What is the process for reporting an issue?"
results = retrieve_documents(question, top_k=5)

print("\n--- Retrieved sources ---")
for item in results:
    print(item["source"], "| score:", round(item["score"], 3))

Guide chunks found: 1

Guide chunk 1:
OpenBridge: Bug Reporting Guide

Before reporting a bug:
1. Search existing issues to check whether the bug
   has already been reported.
2. Check the project's README and CONTRIBUTING guide
   for issue-reporting instructions.
3. If a similar issue exists, add useful details there
   instead of cre

--- Retrieved sources ---
OpenBridge_Bug_Reporting_Guide.md | score: 0.398
CONTRIBUTING.md | score: 0.394
CONTRIBUTING.md | score: 0.298
CONTRIBUTING.md | score: 0.28
CONTRIBUTING.md | score: 0.246


In [ ]:
question_box.submit(
    on_ask,
    inputs=question_box,
    outputs=[answer_md, sources_md]
)


AttributeError: Cannot call submit outside of a gradio.Blocks context.

In [ ]:
import gradio as gr

with gr.Blocks() as demo:
    question_box = gr.Textbox(
        label="Ask a question",
        placeholder="What is the process for reporting an issue?"
    )

    answer_md = gr.Markdown()
    sources_md = gr.Markdown()

    question_box.submit(
        fn=on_ask,
        inputs=[question_box],
        outputs=[answer_md, sources_md]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2dce1c27ea07363f09.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Test Gemma independently
try:
    response = client.models.generate_content(
        model=model,
        contents="Reply with exactly: Hello"
    )
    print(response.text)
except Exception as e:
    print(type(e).__name__)
    print(e)

NameError
name 'model' is not defined


In [ ]:
# Test Gemma independently
import os
from google import genai

client = genai.Client(
    api_key=os.environ.get("GEMINI_API_KEY")
)

model = "gemma-3-27b-flash"

try:
    response = client.models.generate_content(
        model=model,
        contents="Reply with exactly: Hello"
    )
    print(response.text)
except Exception as e:
    print(type(e).__name__)
    print(e)

ClientError
404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemma-3-27b-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}


In [ ]:
from google import genai
import os

client = genai.Client(
    api_key=os.environ.get("GEMINI_API_KEY")
)

for m in client.models.list():
    if "generateContent" in (m.supported_actions or []):
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/l

In [ ]:
from google import genai
import os

client = genai.Client(
    api_key=os.environ.get("GEMINI_API_KEY")
)

response = client.models.generate_content(
    model="gemini-3.1-flash",
    contents="Reply with exactly: Hello"
)

print(response.text)

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-3.1-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [ ]:
from google import genai
import os

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

models = list(client.models.list())

available = [
    m.name for m in models
    if "generateContent" in (m.supported_actions or [])
    and "gemini-3.8-flash" in m.name
]

if not available:
    print("Gemini 3.8 Flash list mein nahi mila.")
    print("Available text-generation models:")
    for m in models:
        if "generateContent" in (m.supported_actions or []):
            print(m.name)
else:
    model = available[0]
    print("Testing model:", model)

    response = client.models.generate_content(
        model=model,
        contents="Reply with exactly: Hello"
    )
    print(response.text)

KeyError: 'GEMINI_API_KEY'

In [ ]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

print("API key loaded:", bool(os.environ["GEMINI_API_KEY"]))


API key loaded: True


In [ ]:
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

if api_key:
    print("API key loaded successfully!")
else:
    print("API key missing. Check Colab Secrets.")

API key loaded successfully!


In [ ]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="models/gemini-3.8-flash".
    contents="Reply with exactly: Hello"
)

print(response.text)

SyntaxError: invalid syntax (2982551695.py, line 9)

In [ ]:
import os
from google.colab import userdata
from google import genai

# Use only the Colab secret we want
api_key = userdata.get("GEMINI_API_KEY")
os.environ.pop("GOOGLE_API_KEY", None)

client = genai.Client(api_key=api_key)

models = list(client.models.list())

available = [
    m.name.removeprefix("models/")
    for m in models
    if "generateContent" in (m.supported_actions or [])
]

print("Available text-generation models:")
for name in available:
    print(name)

Available text-generation models:
gemini-2.5-flash
gemini-2.5-pro
gemini-2.5-flash-preview-tts
gemini-2.5-pro-preview-tts
gemma-4-26b-a4b-it
gemma-4-31b-it
gemini-flash-latest
gemini-flash-lite-latest
gemini-pro-latest
gemini-2.5-flash-lite
gemini-2.5-flash-image
gemini-3-flash-preview
gemini-3.1-pro-preview
gemini-3.1-pro-preview-customtools
gemini-3.1-flash-lite-preview
gemini-3.1-flash-lite
gemini-3-pro-image-preview
gemini-3-pro-image
nano-banana-pro-preview
gemini-3.1-flash-image-preview
gemini-3.1-flash-image
gemini-3.1-flash-lite-image
gemini-nano-banana-2.1
gemini-3.5-flash
gemini-3.5-flash-lite
gemini-omni-flash-preview
gemini-omni-1.1-flash
gemini-3.5-transcribe
gemini-3.6-flash
gemini-3.7-flash
gemini-3.8-flash
gemini-4-argon
lyria-3-clip-preview
lyria-3-pro-preview
lyria-3.5
gemini-3.1-flash-tts-preview
gemini-3.8-flash-tts
gemini-3.8-flash-lite-tts
gemini-robotics-er-2-preview
gemini-2.5-computer-use-preview-10-2025
antigravity-preview-05-2026
antigravity-preview-09-2026
a

In [ ]:
from google.colab import userdata
from google import genai
import os

api_key = userdata.get("GEMINI_API_KEY")
os.environ.pop("GOOGLE_API_KEY", None)

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Reply with exactly: Hello"
)

print(response.text)

Hello


In [ ]:
question_box.submit(
    on_ask,
    inputs=question_box,
    outputs=[answer_md, sources_md]
)

AttributeError: Cannot call submit outside of a gradio.Blocks context.

In [ ]:
question_box.submit(
    on_ask,
    inputs=question_box,
    outputs=[answer_md, sources_md]
)

AttributeError: Cannot call submit outside of a gradio.Blocks context.

## Next steps
1. Re-read your selected problem statement (2A or 2B) in the Theme 2 document.
2. Replace the sample document and example questions with material for your problem statement.
3. Adapt `ANSWER_RULES`, the settings and the application to meet the requirements.
4. Measure the quality of the answers (see "Expected Deliverables" in the problem statement).
5. Attempt items from the **Good to have** section of the problem statement for additional credit.

**Pointers for common extensions:**
- **Markdown and text files (needed for 2B):** read them with `open(path, encoding="utf-8").read()` and treat each file (or each section under a heading) as one page, storing the file name for citations. Allow them in the application with `file_types=[".pdf", ".md", ".txt"]`.
- **Multiple documents:** use `gr.File(file_count="multiple")`, combine the pages of every file, and store the file name in each chunk for citations.
- **Answer without RAG (for comparison):** call `generate(question)` with no context and show the result next to the RAG answer.
- **Relevance threshold:** in `answer()`, if `found[0]["score"]` is below a chosen value, return "The answer could not be found in the document." without calling Gemma.
- **Conversation memory:** replace the question box with `gr.ChatInterface` and include recent messages in the prompt.

## Troubleshooting

| Issue | Resolution |
|---|---|
| The notebook remains in demonstration mode | Add `GOOGLE_API_KEY` in Secrets, enable **Notebook access**, and re-run the Step 2 cell. |
| "API key was rejected" | Create a new key at aistudio.google.com/apikey and update the secret. |
| "Usage limit reached" | Wait about one minute. Free keys allow a limited number of requests per minute. Team members may each create a key. |
| `NameError: name '...' is not defined` | A cell was skipped. Run all cells above in order. |
| "No text was found" | The PDF contains scanned images only. Use a text-based PDF. |
| The public link no longer works | Links expire when the Colab session disconnects. Re-run the launch cell to create a new link. |